In [1]:
import pandas as pd
import re

# ⬇️ Pfad anpassen: falls die Datei z. B. in /data liegt, Pfad entsprechend ergänzen
EXCEL_PATH = "/Users/raffaelruppert/Desktop/SocialMap_Dashboard/data/PLZ_Matching.xlsx"

# 1️⃣ Excel laden
df = pd.read_excel(EXCEL_PATH)

# 2️⃣ Hilfsfunktion, um PLZ-Liste auseinanderzuziehen
def split_plz(s):
    parts = re.split(r"[,\.;]\s*", str(s))     # trennt an Komma, Punkt oder Semikolon
    parts = [p.strip() for p in parts if p.strip()]
    return [p for p in parts if re.fullmatch(r"\d{5}", p)]  # nur 5-stellige PLZ

# 3️⃣ Jede PLZ in eine eigene Zeile bringen
df_long = (
    df.assign(PLZ_List=df["PLZ"].apply(split_plz))
      .explode("PLZ_List", ignore_index=True)
      .rename(columns={"PLZ_List": "PLZ"})
      .drop_duplicates(subset=["Bezirk","Stadtteil","PLZ"])
)

# 4️⃣ Kontrolle: vorher / nachher
print("Zeilen vorher:", len(df))
print("Zeilen nachher:", len(df_long))
df_long.head()

Zeilen vorher: 93
Zeilen nachher: 336


,Bezirk,Stadtteil,PLZ,PLZ
0,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10585
1,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10587
2,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10589
3,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10623
4,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10625


In [3]:
import geopandas as gpd

# Basis-URL des Berliner WFS
BASE_LOR = "https://gdi.berlin.de/services/wfs/lor_2021"

# Direkte GeoJSON-Links für die drei LOR-Ebenen (offiziell, Stand 2021)
urls = {
    "Planungsräume": "https://gdi.berlin.de/services/wfs/lor_2021?service=WFS&version=2.0.0&request=GetFeature&typeNames=lor_2021:a_lor_plr_2021&outputFormat=application/json&srsName=EPSG:25833",
    "Bezirksregionen": "https://gdi.berlin.de/services/wfs/lor_2021?service=WFS&version=2.0.0&request=GetFeature&typeNames=lor_2021:b_lor_bzr_2021&outputFormat=application/json&srsName=EPSG:25833",
    "Prognoseräume": "https://gdi.berlin.de/services/wfs/lor_2021?service=WFS&version=2.0.0&request=GetFeature&typeNames=lor_2021:c_lor_pgr_2021&outputFormat=application/json&srsName=EPSG:25833"
}

lor_plr = gpd.read_file(urls["Planungsräume"])
lor_bzr = gpd.read_file(urls["Bezirksregionen"])
lor_pgr = gpd.read_file(urls["Prognoseräume"])

for name, df in [("Planungsräume", lor_plr), ("Bezirksregionen", lor_bzr), ("Prognoseräume", lor_pgr)]:
    print(f"{name}: {len(df)} Zeilen, Spalten: {list(df.columns)[:5]}")


Planungsräume: 542 Zeilen, Spalten: ['id', 'plr_id', 'plr_name', 'bzr_id', 'bzr_name']
Bezirksregionen: 143 Zeilen, Spalten: ['id', 'bzr_id', 'bzr_name', 'pgr_id', 'pgr_name']
Prognoseräume: 58 Zeilen, Spalten: ['id', 'pgr_id', 'pgr_name', 'bez', 'finhalt']


In [4]:
# --- PLZ-Polygone laden (Berlin) ---
BASE_PLZ = "https://gdi.berlin.de/services/wfs/postleitzahlen"

plz = gpd.read_file(
    BASE_PLZ + "?service=WFS&version=2.0.0&request=GetFeature&typeNames=postleitzahlen:postleitzahlen&outputFormat=application/json&srsName=EPSG:25833"
)

print(f"PLZ-Polygone: {len(plz)} Zeilen, Spalten: {list(plz.columns)[:5]}")

PLZ-Polygone: 193 Zeilen, Spalten: ['id', 'plz', 'finhalt', 'geometry']


In [5]:
# Funktion, um pro PLZ den "besten" (größten Überlappungsanteil) Sozialraum zu finden
def best_match(plz_gdf, lor_gdf, lor_id_col, key_col_plz):
    # Schnittmengen berechnen (welche Flächen sich überschneiden)
    inter = gpd.overlay(plz_gdf[[key_col_plz, "geometry"]], lor_gdf[[lor_id_col, "geometry"]], how="intersection")

    # Fläche der Schnittmenge
    inter["area_intersection"] = inter.geometry.area

    # Fläche der gesamten PLZ
    base = plz_gdf[[key_col_plz, "geometry"]].copy()
    base["area_plz"] = base.geometry.area

    # Verknüpfen
    inter = inter.merge(base.drop(columns="geometry"), on=key_col_plz, how="left")

    # Anteil der Fläche, die überlappt
    inter["share"] = inter["area_intersection"] / inter["area_plz"]

    # Größten Anteil je PLZ auswählen (also: welcher Sozialraum passt am besten)
    best = inter.sort_values("share", ascending=False).drop_duplicates(key_col_plz)

    return best[[key_col_plz, lor_id_col]]

# Mappings berechnen
best_plr = best_match(plz, lor_plr, "plr_id", "plz").rename(columns={"plr_id": "PLR_ID"})
best_bzr = best_match(plz, lor_bzr, "bzr_id", "plz").rename(columns={"bzr_id": "BZR_ID"})
best_pgr = best_match(plz, lor_pgr, "pgr_id", "plz").rename(columns={"pgr_id": "PGR_ID"})

# Zu einer Crosswalk-Tabelle zusammenführen
crosswalk = (
    best_plr.merge(best_bzr, on="plz", how="outer")
            .merge(best_pgr, on="plz", how="outer")
            .rename(columns={"plz": "PLZ"})
)

print("Crosswalk erstellt:", len(crosswalk), "PLZ-Zuordnungen")
crosswalk.head()


Crosswalk erstellt: 193 PLZ-Zuordnungen


,PLZ,PLR_ID,BZR_ID,PGR_ID
0,10115,01100308,011004,0110
1,10117,01100206,011002,0110
2,10119,01100416,011004,0110
3,10178,01100310,011003,0110
4,10179,01100313,011003,0110


In [7]:
# Sicherstellen, dass es nur eine eindeutige PLZ-Spalte gibt
df_long = df_long.loc[:, ~df_long.columns.duplicated()]

# Falls es mehrere PLZ-ähnliche Spalten gibt, prüfen:
print("Spaltennamen in df_long:", list(df_long.columns))

# Und zur Sicherheit auch in der Crosswalk-Tabelle
print("Spaltennamen in crosswalk:", list(crosswalk.columns))

# Merge neu ausführen
out = df_long.merge(crosswalk, on="PLZ", how="left")

# Kontrolle
print("Gesamtzeilen:", len(out))
print("Mit gültiger LOR-Zuordnung:", out['PLR_ID'].notna().sum())

out.head(10)

Spaltennamen in df_long: ['Bezirk', 'Stadtteil', 'PLZ']
Spaltennamen in crosswalk: ['PLZ', 'PLR_ID', 'BZR_ID', 'PGR_ID']
Gesamtzeilen: 336
Mit gültiger LOR-Zuordnung: 0


,Bezirk,Stadtteil,PLZ,PLR_ID,BZR_ID,PGR_ID
0,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",NaN,NaN,NaN
1,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",NaN,NaN,NaN
2,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",NaN,NaN,NaN
3,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",NaN,NaN,NaN
4,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",NaN,NaN,NaN
5,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",NaN,NaN,NaN
6,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",NaN,NaN,NaN
7,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",NaN,NaN,NaN
8,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",NaN,NaN,NaN
9,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",NaN,NaN,NaN


In [8]:
df_long.head(5)

,Bezirk,Stadtteil,PLZ
0,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062..."
1,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062..."
2,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062..."
3,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062..."
4,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062..."


In [9]:
import re
import pandas as pd

# Wenn du df noch im Speicher hast, nutzen wir das direkt:
def split_plz(s):
    # Trenne an Komma, Punkt oder Semikolon
    parts = re.split(r"[,\.;]\s*", str(s))
    parts = [p.strip() for p in parts if p.strip()]
    # Nur echte 5-stellige PLZ behalten
    return [p for p in parts if re.fullmatch(r"\d{5}", p)]

# Hier kommt der entscheidende Schritt: wirklich auseinanderziehen
df_long = (
    df.assign(PLZ_List=df["PLZ"].apply(split_plz))
      .explode("PLZ_List", ignore_index=True)
      .rename(columns={"PLZ_List": "PLZ"})
      .drop_duplicates(subset=["Bezirk", "Stadtteil", "PLZ"])
)

print("Nach Explode:", len(df_long), "Zeilen")
df_long.head(10)


KeyError: 'PLZ'

In [10]:
print(df.columns.tolist())


['id', 'pgr_id', 'pgr_name', 'bez', 'finhalt', 'stand', 'geometry']


In [11]:
import pandas as pd
import re

# ⬇️ Pfad anpassen: falls die Datei z. B. in /data liegt, Pfad entsprechend ergänzen
EXCEL_PATH = "/Users/raffaelruppert/Desktop/SocialMap_Dashboard/data/PLZ_Matching.xlsx"

In [17]:
import pandas as pd
import re

# 1️⃣ Excel einlesen
EXCEL_PATH = "/Users/raffaelruppert/Desktop/SocialMap_Dashboard/data/PLZ_Matching.xlsx"
df_plz = pd.read_excel(EXCEL_PATH)
print(df_plz.columns)

# 2️⃣ Hilfsfunktion, um PLZ-Listen zu trennen
def split_plz(s):
    parts = re.split(r"[,\.;]\s*", str(s))
    parts = [p.strip() for p in parts if p.strip()]
    return [p for p in parts if re.fullmatch(r"\d{5}", p)]

# 3️⃣ Explode – neue Spalte "PLZ_neu" erzeugen
df_long = (
    df_plz.assign(PLZ_neu=df_plz["PLZ"].apply(split_plz))
          .explode("PLZ_neu", ignore_index=True)
          .drop_duplicates(subset=["Bezirk", "Stadtteil", "PLZ_neu"])
)

print("Nach Explode:", len(df_long), "Zeilen")
df_long.head(10)

Index(['Bezirk', 'Stadtteil', 'PLZ'], dtype='object')
Nach Explode: 336 Zeilen


,Bezirk,Stadtteil,PLZ,PLZ_neu
0,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10585
1,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10587
2,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10589
3,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10623
4,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10625
5,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10627
6,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10629
7,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10707
8,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10709
9,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10711


In [19]:
# Merge über PLZ_neu (deine saubere Spalte)
out = df_long.merge(crosswalk, left_on="PLZ_neu", right_on="PLZ", how="left")

# Falls Pandas doppelte Spaltennamen erzeugt hat (z. B. PLZ_x, PLZ_y), sortieren wir das
cols = [c for c in out.columns if not c.endswith("_y")]
out = out[cols]

# Optional: umbenennen für Klarheit
out = out.rename(columns={"PLZ_neu": "PLZ", "PLZ_x": "PLZ_Original"})

# Kontrolle
print("Gesamtzeilen:", len(out))
print("Mit gültiger LOR-Zuordnung:", out['PLR_ID'].notna().sum())
out.head(10)


Gesamtzeilen: 336
Mit gültiger LOR-Zuordnung: 335


,Bezirk,Stadtteil,PLZ_Original,PLZ,PLR_ID,BZR_ID,PGR_ID
0,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10585,04300621,043006,0430
1,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10587,04300620,043006,0430
2,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10589,04300518,043005,0430
3,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10623,04300622,043006,0430
4,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10625,04300624,043006,0430
5,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10627,04300623,043006,0430
6,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10629,04501040,045010,0450
7,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10707,04501043,045010,0450
8,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10709,04500939,045009,0450
9,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10711,04500937,045009,0450


In [21]:
print("lor_plr:", lor_plr.columns.tolist())
print("lor_bzr:", lor_bzr.columns.tolist())
print("lor_pgr:", lor_pgr.columns.tolist())

lor_plr: ['id', 'plr_id', 'plr_name', 'bzr_id', 'bzr_name', 'pgr_id', 'pgr_name', 'bez', 'finhalt', 'stadtraum_id', 'stadtraum_name', 'stand', 'geometry']
lor_bzr: ['id', 'bzr_id', 'bzr_name', 'pgr_id', 'pgr_name', 'bez', 'finhalt', 'stand', 'geometry']
lor_pgr: ['id', 'pgr_id', 'pgr_name', 'bez', 'finhalt', 'stand', 'geometry']


In [22]:
# 1️⃣ Relevante Zuordnungstabellen vorbereiten
plr_names = lor_plr[["plr_id", "plr_name", "bzr_id", "bzr_name"]].drop_duplicates()
bzr_names = lor_bzr[["bzr_id", "bzr_name", "pgr_id", "pgr_name"]].drop_duplicates()
pgr_names = lor_pgr[["pgr_id", "pgr_name", "bez"]].drop_duplicates()

# 2️⃣ Merge mit klaren Suffixen, damit nichts verschluckt wird
out_named = (
    out
    .merge(plr_names, left_on="PLR_ID", right_on="plr_id", how="left", suffixes=("", "_plr"))
    .merge(bzr_names, left_on="BZR_ID", right_on="bzr_id", how="left", suffixes=("", "_bzr"))
    .merge(pgr_names, left_on="PGR_ID", right_on="pgr_id", how="left", suffixes=("", "_pgr"))
)

# 3️⃣ Nur relevante Spalten auswählen
out_named = out_named[[
    "Bezirk", "Stadtteil", "PLZ_Original", "PLZ",
    "PLR_ID", "plr_name",
    "BZR_ID", "bzr_name",
    "PGR_ID", "pgr_name",
    "bez"
]]

# 4️⃣ Spalten schöner benennen
out_named = out_named.rename(columns={
    "plr_name": "Planungsraum_Name",
    "bzr_name": "Bezirksregion_Name",
    "pgr_name": "Prognoseraum_Name",
    "bez": "Bezirk_Name"
})

# Kontrolle
print("✅ Ergebnis fertig. Spalten:", list(out_named.columns))
out_named.head(10)


✅ Ergebnis fertig. Spalten: ['Bezirk', 'Stadtteil', 'PLZ_Original', 'PLZ', 'PLR_ID', 'Planungsraum_Name', 'BZR_ID', 'Bezirksregion_Name', 'PGR_ID', 'Prognoseraum_Name', 'Bezirk_Name']


,Bezirk,Stadtteil,PLZ_Original,PLZ,PLR_ID,Planungsraum_Name,BZR_ID,Bezirksregion_Name,PGR_ID,Prognoseraum_Name,Bezirk_Name
0,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10585,04300621,Richard-Wagner-Straße,043006,Otto-Suhr-Allee/Kantstraße,0430,Charlottenburg Zentrum,04 - Charlottenburg-Wilmersdorf
1,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10587,04300620,Spreestadt,043006,Otto-Suhr-Allee/Kantstraße,0430,Charlottenburg Zentrum,04 - Charlottenburg-Wilmersdorf
2,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10589,04300518,Kaiserin-Augusta-Allee,043005,Mierendorffplatz,0430,Charlottenburg Zentrum,04 - Charlottenburg-Wilmersdorf
3,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10623,04300622,Ernst-Reuter-Platz,043006,Otto-Suhr-Allee/Kantstraße,0430,Charlottenburg Zentrum,04 - Charlottenburg-Wilmersdorf
4,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10625,04300624,Savignyplatz,043006,Otto-Suhr-Allee/Kantstraße,0430,Charlottenburg Zentrum,04 - Charlottenburg-Wilmersdorf
5,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10627,04300623,Karl-August-Platz,043006,Otto-Suhr-Allee/Kantstraße,0430,Charlottenburg Zentrum,04 - Charlottenburg-Wilmersdorf
6,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10629,04501040,Hindemithplatz,045010,Lietzenburger Straße,0450,Wilmersdorf Zentrum,04 - Charlottenburg-Wilmersdorf
7,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10707,04501043,Preußenpark,045010,Lietzenburger Straße,0450,Wilmersdorf Zentrum,04 - Charlottenburg-Wilmersdorf
8,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10709,04500939,Eisenzahnstraße,045009,Halensee,0450,Wilmersdorf Zentrum,04 - Charlottenburg-Wilmersdorf
9,Charlottenburg-Wilmersdorf,Charlottenburg,"10585, 10587, 10589, 10623, 10625. 10627, 1062...",10711,04500937,Halensee,045009,Halensee,0450,Wilmersdorf Zentrum,04 - Charlottenburg-Wilmersdorf


In [23]:
# Überblick über NaN-Werte in allen wichtigen Spalten
qa_summary = (
    out_named[["PLR_ID", "Planungsraum_Name", 
                "BZR_ID", "Bezirksregion_Name", 
                "PGR_ID", "Prognoseraum_Name", 
                "Bezirk_Name"]]
    .isna()
    .sum()
    .rename("NaN_Count")
    .to_frame()
)

qa_summary["%_Fehlend"] = (qa_summary["NaN_Count"] / len(out_named) * 100).round(2)
print("🔍 Übersicht fehlender Werte:")
display(qa_summary)

# Falls du zusätzlich sehen willst, welche Zeilen betroffen sind:
fehlende = out_named[
    out_named[["PLR_ID", "BZR_ID", "PGR_ID"]].isna().any(axis=1)
]
print(f"\n🚨 Zeilen mit unvollständiger Zuordnung: {len(fehlende)} von {len(out_named)}")
fehlende.head(10)

🔍 Übersicht fehlender Werte:


,NaN_Count,%_Fehlend
PLR_ID,1,0.3
Planungsraum_Name,1,0.3
BZR_ID,1,0.3
Bezirksregion_Name,1,0.3
PGR_ID,1,0.3
Prognoseraum_Name,1,0.3
Bezirk_Name,1,0.3



🚨 Zeilen mit unvollständiger Zuordnung: 1 von 336


,Bezirk,Stadtteil,PLZ_Original,PLZ,PLR_ID,Planungsraum_Name,BZR_ID,Bezirksregion_Name,PGR_ID,Prognoseraum_Name,Bezirk_Name
290,Tempelhof-Schöneberg,Schöneberg,"10777, 10778, 10779, 10781, 10783, 10785, 1078...",10778,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
out_named.loc[out_named["PLZ"] == "10778", ["PLR_ID", "Planungsraum_Name", "BZR_ID", "Bezirksregion_Name", "PGR_ID", "Prognoseraum_Name", "Bezirk_Name"]] = [
    "07010209", "Winterfeldtplatz",  # Planungsraum-ID & Name
    "070102", "Schöneberg Nord",     # Bezirksregion-ID & Name
    "0701", "Schöneberg Nord",       # Prognoseraum-ID & Name
    "07 - Tempelhof-Schöneberg"      # Bezirk
]

In [25]:
# Überblick über NaN-Werte in allen wichtigen Spalten
qa_summary = (
    out_named[["PLR_ID", "Planungsraum_Name", 
                "BZR_ID", "Bezirksregion_Name", 
                "PGR_ID", "Prognoseraum_Name", 
                "Bezirk_Name"]]
    .isna()
    .sum()
    .rename("NaN_Count")
    .to_frame()
)

qa_summary["%_Fehlend"] = (qa_summary["NaN_Count"] / len(out_named) * 100).round(2)
print("🔍 Übersicht fehlender Werte:")
display(qa_summary)

# Falls du zusätzlich sehen willst, welche Zeilen betroffen sind:
fehlende = out_named[
    out_named[["PLR_ID", "BZR_ID", "PGR_ID"]].isna().any(axis=1)
]
print(f"\n🚨 Zeilen mit unvollständiger Zuordnung: {len(fehlende)} von {len(out_named)}")
fehlende.head(10)

🔍 Übersicht fehlender Werte:


,NaN_Count,%_Fehlend
PLR_ID,0,0.0
Planungsraum_Name,0,0.0
BZR_ID,0,0.0
Bezirksregion_Name,0,0.0
PGR_ID,0,0.0
Prognoseraum_Name,0,0.0
Bezirk_Name,0,0.0



🚨 Zeilen mit unvollständiger Zuordnung: 0 von 336


,Bezirk,Stadtteil,PLZ_Original,PLZ,PLR_ID,Planungsraum_Name,BZR_ID,Bezirksregion_Name,PGR_ID,Prognoseraum_Name,Bezirk_Name


In [26]:
out_named.groupby("Bezirk_Name")["PLZ"].nunique().sort_values(ascending=False)

Bezirk_Name
04 - Charlottenburg-Wilmersdorf    25
07 - Tempelhof-Schöneberg          23
01 - Mitte                         19
03 - Pankow                        17
06 - Steglitz-Zehlendorf           17
08 - Neukölln                      16
12 - Reinickendorf                 14
09 - Treptow-Köpenick              14
11 - Lichtenberg                   12
05 - Spandau                       12
02 - Friedrichshain-Kreuzberg      11
10 - Marzahn-Hellersdorf           11
Name: PLZ, dtype: int64

In [27]:
SAVE_PATH = "/Users/raffaelruppert/Desktop/SocialMap_Dashboard/data/berlin_plz_to_sozialraum.xlsx"
out_named.to_excel(SAVE_PATH, index=False)
print(f"✅ Datei gespeichert unter: {SAVE_PATH}")

✅ Datei gespeichert unter: /Users/raffaelruppert/Desktop/SocialMap_Dashboard/data/berlin_plz_to_sozialraum.xlsx
